In [1]:
import os
from pathlib import Path
from eodag import EODataAccessGateway, setup_logging
from shapely.geometry import box
import zipfile

In [3]:
import eodag
print(eodag.__version__)

4.0.2


In [2]:
# 1. LOGGING
setup_logging(verbose=2)


# 2. AUTHENTIFICATION
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__USERNAME"] = "damba.kone@umontpellier.fr"
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__PASSWORD"] = "0767991488Dk@"

In [5]:
# 3. INIT EODAG
dag = EODataAccessGateway()
dag.set_preferred_provider("cop_dataspace")

# 4. DOSSIERS
output_dir = Path("data/raw/Sentinel1")
zip_dir = output_dir / "zip"
extract_dir = output_dir / "SAFE"

zip_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)

# 5. ROI (Cayenne)
roi = box(-53.5, 4.5, -51.5, 6.5)

# 6. RECHERCHE Sentinel-1 GRD
print(" Recherche des images Sentinel-1 GRD...")

search_results = dag.search(
    collection="SENTINEL-1",
    productType="GRD",   
    geom=roi,
    start="2023-01-01",
    end="2023-12-31",
    limit=5,
    provider="cop_dataspace"
)

print(f" {len(search_results)} images trouvées")

2026-04-08 15:08:22,596 eodag.provider                   [INFO    ] Loading user configuration from: C:\Users\Kone\.config\eodag\eodag.yml
2026-04-08 15:08:22,599 eodag.provider                   [WARNING ] providers: skipped creating due to invalid config
2026-04-08 15:08:22,616 eodag.core                       [INFO    ] usgs: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 15:08:22,617 eodag.core                       [INFO    ] aws_eos: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 15:08:22,618 eodag.core                       [INFO    ] cop_ads: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 15:08:22,618 eodag.core                       [INFO    ] cop_cds: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 15:08:22,619 eodag.core                       [INFO    ] meteoblue: provider ne

 Recherche des images Sentinel-1 GRD...
 5 images trouvées


In [6]:
# 7. APERÇU
print("\n--- Aperçu des images ---")

for p in search_results:
    print(p.properties.get("title"))
    print("Date:", p.properties.get("start_datetime"))
    print("Polarisation:", p.properties.get("polarizationMode"))
    print("-" * 40)


--- Aperçu des images ---
S1A_IW_GRDH_1SDV_20230101T091228_20230101T091257_046588_05954E_3DB1
Date: 2023-01-01T09:12:28.801000Z
Polarisation: None
----------------------------------------
S1A_IW_GRDH_1SDV_20230101T091257_20230101T091322_046588_05954E_ACB0
Date: 2023-01-01T09:12:57.832000Z
Polarisation: None
----------------------------------------
S1A_S2_GRDH_1SDH_20230101T213533_20230101T213557_046596_05958E_99EB
Date: 2023-01-01T21:35:33.561000Z
Polarisation: None
----------------------------------------
S1A_IW_GRDH_1SDV_20230106T092034_20230106T092059_046661_0597BB_562C
Date: 2023-01-06T09:20:34.288000Z
Polarisation: None
----------------------------------------
S1A_IW_GRDH_1SDV_20230106T092059_20230106T092124_046661_0597BB_562E
Date: 2023-01-06T09:20:59.288000Z
Polarisation: None
----------------------------------------


In [7]:
# 8. TELECHARGEMENT
print("\n Téléchargement des ZIP...")

downloaded_files = []

for i, product in enumerate(search_results):
    try:
        print(f"[{i+1}/{len(search_results)}] {product.properties.get('title')}")

        path = dag.download(
            product,
            outputs_prefix=str(zip_dir),
            extract=False   
        )

        downloaded_files.append(path)

        print(" OK téléchargé :", path)

    except Exception as e:
        print(" Erreur téléchargement :", e)


 Téléchargement des ZIP...
[1/5] S1A_IW_GRDH_1SDV_20230101T091228_20230101T091257_046588_05954E_3DB1


0.00B [00:00, ?B/s]

2026-04-08 15:09:12,341 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(c51f6152-7c4c-5327-a3df-a0bed17a59e9)/$value
2026-04-08 15:15:08,139 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 15:15:08,143 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(c51f6152-7c4c-5327-a3df-a0bed17a59e9)/$value


 OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S1A_IW_GRDH_1SDV_20230101T091228_20230101T091257_046588_05954E_3DB1.zip
[2/5] S1A_IW_GRDH_1SDV_20230101T091257_20230101T091322_046588_05954E_ACB0


0.00B [00:00, ?B/s]

...

2026-04-08 15:35:59,206 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(10194e56-c968-5aac-bb2e-d440d273cb94)/$value
2026-04-08 15:56:25,846 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 15:56:25,849 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(10194e56-c968-5aac-bb2e-d440d273cb94)/$value


 OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S1A_IW_GRDH_1SDV_20230106T092034_20230106T092059_046661_0597BB_562C.zip
[5/5] S1A_IW_GRDH_1SDV_20230106T092059_20230106T092124_046661_0597BB_562E


0.00B [00:00, ?B/s]

2026-04-08 15:56:26,684 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(acb3c319-dc2e-58cd-af40-798a54340fd4)/$value
2026-04-08 16:18:49,171 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 16:18:49,174 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(acb3c319-dc2e-58cd-af40-798a54340fd4)/$value


 OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S1A_IW_GRDH_1SDV_20230106T092059_20230106T092124_046661_0597BB_562E.zip


In [8]:
# 9. EXTRACTION
print("\n Extraction des ZIP...")

for zip_path in downloaded_files:
    try:
        zip_path = Path(zip_path)

        print(f"Extraction : {zip_path.name}")

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print(" OK extrait")

    except Exception as e:
        print(" Erreur extraction :", e)

print("\n Pipeline Sentinel-1 terminé proprement !")


 Extraction des ZIP...
Extraction : S1A_IW_GRDH_1SDV_20230101T091228_20230101T091257_046588_05954E_3DB1.zip
 OK extrait
Extraction : S1A_IW_GRDH_1SDV_20230101T091257_20230101T091322_046588_05954E_ACB0.zip
 OK extrait
Extraction : S1A_S2_GRDH_1SDH_20230101T213533_20230101T213557_046596_05958E_99EB.zip
 OK extrait
Extraction : S1A_IW_GRDH_1SDV_20230106T092034_20230106T092059_046661_0597BB_562C.zip
 OK extrait
Extraction : S1A_IW_GRDH_1SDV_20230106T092059_20230106T092124_046661_0597BB_562E.zip
 OK extrait

 Pipeline Sentinel-1 terminé proprement !
